# Sahayak - ORBIT pipeline (one-shot, runs unattended)
Run with **Save Version -> Save & Run All (Commit)**. It runs in the background (you can close the tab), takes about 3 hours and keeps its outputs
(`sahayak2/reports`, `sahayak2/checkpoints`) in the version's **Output** tab. Interactive sessions get killed when idle; committed runs do not.
Notebook settings needed first: Accelerator **GPU T4 x2**, Internet **On**.
Order: download+convert -> main model C -> cross-video model -> diagnosis (val split) -> LoRA -> baselines -> plot.
Every step is a separate command, so one failure does not stop the rest; check the logs in `reports/log_*.txt`.


In [ ]:
import os, subprocess, zipfile, re, shutil, requests
os.chdir('/kaggle/working')
!rm -rf sahayak2
!git clone -q https://github.com/shreyaaaah/fewshot-object-detection-for-visually-impaired-person.git sahayak2
os.chdir('/kaggle/working/sahayak2')
!pip -q install timm scikit-learn tqdm pyyaml
os.makedirs('reports', exist_ok=True); os.makedirs('/kaggle/temp', exist_ok=True)
RAW, ROOT = '/kaggle/temp/orbit_raw', '/kaggle/temp/orbit_instances'      # temporary (not saved in the output)
EPS = 1500
EV = f'--root {ROOT} --split test --episodes 200 --k-shots 1 3 5 10 --cross-video'
!nvidia-smi | head -12


In [ ]:
files = requests.get('https://api.figshare.com/v2/articles/14294597', timeout=60).json()['files']
byname = {f['name']: f for f in files}
os.makedirs(RAW, exist_ok=True)
EVERY, TMP = 10, '/kaggle/temp/'

def fetch(name):
    f = byname[name]; z = TMP + name
    print('downloading', name, round(f['size'] / 1e9, 1), 'GB ...', flush=True)
    subprocess.run(['wget', '-q', '-c', '-O', z, f['download_url']], check=True)
    kept = total = 0
    pat = re.compile(r'(\d+)\.(?:jpe?g|png)$', re.I)
    with zipfile.ZipFile(z) as zf:
        for m in zf.infolist():
            if m.is_dir():
                continue
            total += 1
            k = pat.search(m.filename)
            if k and int(k.group(1)) % EVERY != 0:
                continue
            zf.extract(m, RAW); kept += 1
    os.remove(z)
    print(f'{name}: kept {kept} of {total} frames', flush=True)

fetch('validation_224.zip')
fetch('train_224.zip')
!python tools/prepare_orbit.py --src {RAW} --out {ROOT} --every 1 2>&1 | tail -5
shutil.rmtree(RAW, ignore_errors=True)
n = len(os.listdir(ROOT)) if os.path.isdir(ROOT) else 0
print(n, 'object folders in ROOT (expect about 330)')
assert n > 100, 'conversion found too little data'


## A. Main model C (last 2 blocks, open-set loss) + test evaluation

In [ ]:
!python -m src.train.train --root {ROOT} --mode partial --loss open --episodes {EPS} --tag C_partial_open --amp 2>&1 | tee reports/log_train_C_partial_open.txt | tail -5
!python -m src.eval.episodic {EV} --ckpt checkpoints/embedder_C_partial_open.pt --tag C_partial_open 2>&1 | tee reports/log_eval_C_partial_open.txt | tail -45


## B. Cross-video model (episodes follow the test protocol: support from clean, queries from clutter)

In [ ]:
!python -m src.train.train --root {ROOT} --mode partial --loss open --episodes {EPS} --tag C_partial_open_xv --cross-video --amp 2>&1 | tee reports/log_train_C_partial_open_xv.txt | tail -5
!python -m src.eval.episodic {EV} --ckpt checkpoints/embedder_C_partial_open_xv.pt --tag C_partial_open_xv 2>&1 | tee reports/log_eval_C_partial_open_xv.txt | tail -45


## C. Memory diagnosis on the VALIDATION split only (read-only)

In [ ]:
!python tools/diagnose_memory.py --root {ROOT} --split val --frozen-backbone vit_small_patch14_dinov2.lvd142m --episodes 30 2>&1 | tee reports/log_diag_B_frozen.txt
!python tools/diagnose_memory.py --root {ROOT} --split val --ckpt checkpoints/embedder_C_partial_open.pt --episodes 30 2>&1 | tee reports/log_diag_C.txt
!python tools/diagnose_memory.py --root {ROOT} --split val --ckpt checkpoints/embedder_C_partial_open_xv.pt --episodes 30 2>&1 | tee reports/log_diag_C_xv.txt


## D. LoRA variant

In [ ]:
!python -m src.train.train --root {ROOT} --mode lora --loss open --episodes {EPS} --tag D_lora_open --amp 2>&1 | tee reports/log_train_D_lora_open.txt | tail -5
!python -m src.eval.episodic {EV} --ckpt checkpoints/embedder_D_lora_open.pt --tag D_lora_open 2>&1 | tee reports/log_eval_D_lora_open.txt | tail -45


## E. Baselines (for the comparison plot)

In [ ]:
!python -m src.eval.episodic {EV} --frozen-backbone hsv_hist --tag A0_hsv_hist 2>&1 | tail -3
!python -m src.eval.episodic {EV} --frozen-backbone resnet50.a1_in1k --tag A_resnet50_frozen 2>&1 | tail -3
!python -m src.eval.episodic {EV} --tag B_dinov2_frozen 2>&1 | tail -3


## F. Plot and table

In [ ]:
import json, glob, pathlib
import matplotlib.pyplot as plt
R = {}
for f in sorted(glob.glob('reports/episodic_*_test.json')):
    R[pathlib.Path(f).stem.replace('episodic_', '').replace('_test', '')] = json.load(open(f))['results']
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for name, res in R.items():
    ks = sorted(int(k) for k in res)
    for a, key, scale, ttl in ((ax[0], 'closed_acc', 100, 'Closed-set accuracy (%) vs shots K'),
                               (ax[1], 'auroc_ours_prob', 1, 'Open-set AUROC vs K')):
        a.errorbar(ks, [res[str(k)][key][0] * scale for k in ks], yerr=[res[str(k)][key][1] * scale for k in ks],
                   marker='o', capsize=3, label=name)
        a.set_title(ttl); a.set_xlabel('K (examples per object)'); a.grid(alpha=.3)
ax[0].legend(fontsize=7)
plt.tight_layout(); plt.savefig('reports/orbit_results.png', dpi=150)
print(sorted(os.listdir('reports'))); print(sorted(os.listdir('checkpoints')))
